# API Football Code
### Introduction
For the work on proving Bolivian teams outplay their opponents due to altitude difference I need to find
- All Libertadores fixtures including Bolivian teams vs Non-Bolivian Teams
- Venue of home and away leg
- Altitude of venue
- Scores

I think most of this can be done from the fixtures endpoint. It will be especially interesting to contrast first half and second half performance

In [20]:
import json
import http.client
import os
from dotenv import load_dotenv
import requests
import time
import pandas as pd

In [3]:
load_dotenv()
FB_API_KEY = os.getenv("API_FOOTBALL_KEY")

In [6]:
# setup API Connection
conn = http.client.HTTPSConnection("v3.football.api-sports.io")

HEADERS = {
    "Accept":           "application/json",
    "x-apisports-key":  FB_API_KEY,
}

conn.request("GET", "/fixtures?id=215662", headers=HEADERS)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"get":"fixtures","parameters":{"id":"215662"},"errors":[],"results":1,"paging":{"current":1,"total":1},"response":[{"fixture":{"id":215662,"referee":"H. Mastr\u00e1ngelo","timezone":"UTC","date":"2019-10-20T14:00:00+00:00","timestamp":1571580000,"periods":{"first":1571580000,"second":1571583600},"venue":{"id":33,"name":"Estadio Jos\u00e9 Mar\u00eda Minella","city":"Mar del Plata, Provincia de Buenos Aires"},"status":{"long":"Match Finished","short":"FT","elapsed":90,"extra":null}},"league":{"id":128,"name":"Liga Profesional Argentina","country":"Argentina","logo":"https:\/\/media.api-sports.io\/football\/leagues\/128.png","flag":"https:\/\/media.api-sports.io\/flags\/ar.svg","season":2019,"round":"Regular Season - 10","standings":true},"teams":{"home":{"id":463,"name":"Aldosivi","logo":"https:\/\/media.api-sports.io\/football\/teams\/463.png","winner":true},"away":{"id":442,"name":"Defensa Y Justicia","logo":"https:\/\/media.api-sports.io\/football\/teams\/442.png","winner":false}},"g

In [32]:
# code from football API docs example
# ——— CONFIG ———
API_KEY    = FB_API_KEY
BASE_URL   = "https://v3.football.api-sports.io/fixtures"
HEADERS = {
    "Accept":           "application/json",
    "x-apisports-key":  API_KEY,
}
LEAGUE_ID  = 13     # e.g. Premier League
SEASON     = 2021   # e.g. 2024
# Only completed ("FT","AET","PEN") + in‑progress ("1H","HT","2H","ET","BT","P")
STATUS     = "FT"
CHUNK_SIZE = 20     # max fixture IDs per call :contentReference[oaicite:2]{index=2}

In [33]:
def chunk_list(lst, size):
    """Split list into chunks of at most `size`."""
    return [lst[i:i+size] for i in range(0, len(lst), size)]

def fetch_fixture_ids():
    """Return a list of fixture IDs for the given league/season/status."""
    params = {
        "league": LEAGUE_ID,
        "season": SEASON,
        "status": STATUS
    }
    resp = requests.get(BASE_URL, headers=HEADERS, params=params)
    print(resp.status_code, resp.headers.get("content-type"))
    #print(resp.text)
    resp.raise_for_status()
    fixtures = resp.json().get("response", [])
    return [f["fixture"]["id"] for f in fixtures if "fixture" in f]

In [34]:
fixture_ids = fetch_fixture_ids()
chunks = chunk_list(fixture_ids, CHUNK_SIZE)
print(f"Will retrieve {len(fixture_ids)} fixtures in {len(chunks)} batches.")


200 application/json
Will retrieve 149 fixtures in 8 batches.


In [35]:
def fetch_fixtures_by_ids(id_group):
    """Fetch full fixture data for a hyphen‑joined group of IDs."""
    ids_param = "-".join(str(i) for i in id_group)
    resp = requests.get(
        BASE_URL,
        headers=HEADERS,
        params={"ids": ids_param}
    )
    resp = requests.get(BASE_URL, headers=HEADERS, params={"ids": ids_param})
    print(resp.status_code, resp.request.url)
    #print(resp.text)   # or resp.json()
    resp.raise_for_status()
    return resp.json().get("response", [])

all_data_2021 = []
for group in chunks:
    batch = fetch_fixtures_by_ids(group)
    all_data_2022.extend(batch)
    time.sleep(1)  # crude rate‑limit guard; adjust as needed :contentReference[oaicite:3]{index=3}

print(f"Retrieved detailed data for {len(all_data_2023)} fixtures.")

200 https://v3.football.api-sports.io/fixtures?ids=819350-819348-819352-819351-819353-819340-819344-842760-828392-819342-842762-819346-842764-819341-842761-828393-819343-819347-842763-842765
200 https://v3.football.api-sports.io/fixtures?ids=846333-846336-846331-846335-846334-846337-853677-853676-853675-853679-853678-853680-853682-853683-853681-853685-853686-853684-853687-853688
200 https://v3.football.api-sports.io/fixtures?ids=853689-853690-853691-853692-853693-853694-853696-853695-853697-853698-853702-853699-853701-853700-853703-853704-853705-853706-853709-853710
200 https://v3.football.api-sports.io/fixtures?ids=853708-853707-853712-853713-853711-853714-853718-853716-853715-853717-853719-853720-853721-853722-853723-853724-853725-853728-853727-853726
200 https://v3.football.api-sports.io/fixtures?ids=853729-853730-853731-853733-853732-853734-853735-853736-853737-853738-853740-853741-853739-853743-853742-853744-853745-853746-853749-853747
200 https://v3.football.api-sports.io/fixture

In [36]:
all_data_2022[1]

{'fixture': {'id': 819341,
  'referee': 'G. Vargas',
  'timezone': 'UTC',
  'date': '2022-02-28T22:15:00+00:00',
  'timestamp': 1646086500,
  'periods': {'first': 1646086500, 'second': 1646090100},
  'venue': {'id': 1660,
   'name': 'Estadio Monumental de Maturín',
   'city': 'Maturín'},
  'status': {'long': 'Match Finished',
   'short': 'FT',
   'elapsed': 90,
   'extra': None}},
 'league': {'id': 13,
  'name': 'CONMEBOL Libertadores',
  'country': 'World',
  'logo': 'https://media.api-sports.io/football/leagues/13.png',
  'flag': None,
  'season': 2022,
  'round': '2nd Round',
  'standings': True},
 'teams': {'home': {'id': 2811,
   'name': 'Monagas SC',
   'logo': 'https://media.api-sports.io/football/teams/2811.png',
   'winner': True},
  'away': {'id': 2325,
   'name': 'Everton de Vina',
   'logo': 'https://media.api-sports.io/football/teams/2325.png',
   'winner': False}},
 'goals': {'home': 1, 'away': 0},
 'score': {'halftime': {'home': 0, 'away': 0},
  'fulltime': {'home': 1, '

In [18]:
# pick out the relevant columns for my table

competition_name = all_data[i]['league']['id']
competition_id = all_data[i]['league']['name']
competition_season = all_data[i]['league']['season']
competition_round = all_data[i]['league']['round']
fixture_id = all_data[i]['fixture']['id']
fixture_date = all_data[i]['fixture']['date']
fixture_venue_id = all_data[i]['fixture']['venue']['id']
fixture_venue_name = all_data[i]['fixture']['venue']['name']
fixture_venue_city = all_data[i]['fixture']['venue']['city']
home_team_id = all_data[i]['teams']['home']['id']
home_team_name = all_data[i]['teams']['home']['name']
home_team_h1_goals = all_data[i]['score']['halftime']['home']
home_team_h2_goals = all_data[i]['score']['halftime']['home'] - all_data[i]['teams']['score']['halftime']['home']
home_team_ft_goals = all_data[i]['score']['fulltime']['home']
away_team_id = all_data[i]['teams']['away']['id']
away_team_name = all_data[i]['teams']['away']['name']
away_team_h1_goals = all_data[i]['score']['halftime']['away']
away_team_h2_goals = all_data[i]['score']['halftime']['away'] - all_data[i]['teams']['score']['halftime']['away']
away_team_ft_goals = all_data[i]['score']['fulltime']['away']


NameError: name 'i' is not defined

In [37]:


# Assuming all_data is your list of JSON objects
rows = []

for match in all_data_2023:
    row = {
        'competition_id': match['league']['id'],
        'competition_name': match['league']['name'],
        'competition_season': match['league']['season'],
        'competition_round': match['league']['round'],
        'fixture_id': match['fixture']['id'],
        'fixture_date': match['fixture']['date'],
        'fixture_venue_id': match['fixture']['venue']['id'],
        'fixture_venue_name': match['fixture']['venue']['name'],
        'fixture_venue_city': match['fixture']['venue']['city'],
        'home_team_id': match['teams']['home']['id'],
        'home_team_name': match['teams']['home']['name'],
        'home_team_h1_goals': match['score']['halftime']['home'],
        'home_team_h2_goals': match['score']['fulltime']['home'] - match['score']['halftime']['home'],
        'home_team_ft_goals': match['score']['fulltime']['home'],
        'away_team_id': match['teams']['away']['id'],
        'away_team_name': match['teams']['away']['name'],
        'away_team_h1_goals': match['score']['halftime']['away'],
        'away_team_h2_goals': match['score']['fulltime']['away'] - match['score']['halftime']['away'],
        'away_team_ft_goals': match['score']['fulltime']['away']
    }
    rows.append(row)

# Convert to DataFrame
df_22 = pd.DataFrame(rows)


In [22]:
df

,competition_id,competition_name,competition_season,competition_round,fixture_id,fixture_date,fixture_venue_id,fixture_venue_name,fixture_venue_city,home_team_id,home_team_name,home_team_h1_goals,home_team_h2_goals,home_team_ft_goals,away_team_id,away_team_name,away_team_h1_goals,away_team_h2_goals,away_team_ft_goals
0,13,CONMEBOL Libertadores,2024,2nd Round,1149590,2024-02-21T00:30:00+00:00,380.0,Estadio Atanasio Girardot,Medellín,1144,Rionegro Aguilas,0,0,0,794,RB Bragantino,0,0,0
1,13,CONMEBOL Libertadores,2024,2nd Round,1149592,2024-02-21T00:30:00+00:00,2487.0,Estadio Municipal El Alto,El Alto,3700,Always Ready,1,5,6,2546,Sporting Cristal,1,0,1
2,13,CONMEBOL Libertadores,2024,2nd Round,1149593,2024-02-28T00:30:00+00:00,1228.0,Estadio Nacional de Lima,Lima,2546,Sporting Cristal,1,2,3,3700,Always Ready,1,0,1
3,13,CONMEBOL Libertadores,2024,2nd Round,1149594,2024-02-23T00:30:00+00:00,80.0,Estadio Malvinas Argentinas,"Mendoza, Provincia de Mendoza",439,Godoy Cruz,0,0,0,2315,Colo Colo,0,1,1
4,13,CONMEBOL Libertadores,2024,2nd Round,1149595,2024-03-01T00:30:00+00:00,319.0,Estadio Monumental David Arellano,Santiago de Chile,2315,Colo Colo,0,0,0,439,Godoy Cruz,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,13,CONMEBOL Libertadores,2024,Semi-finals,1308770,2024-10-23T00:30:00+00:00,21427.0,Arena MRV,"Belo Horizonte, Minas Gerais",1062,Atletico-MG,1,2,3,435,River Plate,0,0,0
145,13,CONMEBOL Libertadores,2024,Semi-finals,1308771,2024-10-24T00:30:00+00:00,218.0,Estádio Nilton Santos,Rio de Janeiro,120,Botafogo,0,5,5,2348,Penarol,0,0,0
146,13,CONMEBOL Libertadores,2024,Semi-finals,1308772,2024-10-30T00:30:00+00:00,19570.0,Estadio Mâs Monumental,"Capital Federal, Ciudad de Buenos Aires",435,River Plate,0,0,0,1062,Atletico-MG,0,0,0
147,13,CONMEBOL Libertadores,2024,Semi-finals,1308773,2024-10-31T00:30:00+00:00,1624.0,Estadio Centenario,Montevideo,2348,Penarol,1,2,3,120,Botafogo,0,1,1


In [31]:
df_23

,competition_id,competition_name,competition_season,competition_round,fixture_id,fixture_date,fixture_venue_id,fixture_venue_name,fixture_venue_city,home_team_id,home_team_name,home_team_h1_goals,home_team_h2_goals,home_team_ft_goals,away_team_id,away_team_name,away_team_h1_goals,away_team_h2_goals,away_team_ft_goals
0,13,CONMEBOL Libertadores,2023,2nd Round,981590,2023-02-22T00:00:00+00:00,319.0,Estadio Monumental David Arellano,Santiago de Chile,2316,Curico Unido,0,0,0,1176,Cerro Porteno,0,1,1
1,13,CONMEBOL Libertadores,2023,2nd Round,981591,2023-02-27T22:00:00+00:00,1214.0,Estadio General Pablo Rojas,Asunción,1176,Cerro Porteno,0,1,1,2316,Curico Unido,0,0,0
2,13,CONMEBOL Libertadores,2023,2nd Round,981592,2023-02-23T00:30:00+00:00,1653.0,Estadio Olímpico de la UCV,Caracas,2810,Carabobo FC,0,0,0,1062,Atletico-MG,0,0,0
3,13,CONMEBOL Libertadores,2023,2nd Round,981593,2023-03-02T00:30:00+00:00,234.0,Estádio Governador Magalhães Pinto,"Belo Horizonte, Minas Gerais",1062,Atletico-MG,2,1,3,2810,Carabobo FC,1,0,1
4,13,CONMEBOL Libertadores,2023,2nd Round,981594,2023-02-22T22:00:00+00:00,337.0,Estadio El Teniente,Rancagua,2336,Magallanes,1,2,3,3700,Always Ready,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,13,CONMEBOL Libertadores,2023,Quarter-finals,1115603,2023-09-01T00:30:00+00:00,11622.0,Estadio Defensores del Chaco,Asunción,1182,Olimpia,1,0,1,124,Fluminense,1,2,3
145,13,CONMEBOL Libertadores,2023,Quarter-finals,1115604,2023-08-24T00:30:00+00:00,46.0,Estadio Alberto José Armando,Ciudad de Buenos Aires,451,Boca Juniors,0,0,0,436,Racing Club,0,0,0
146,13,CONMEBOL Libertadores,2023,Semi-finals,1125015,2023-09-29T00:30:00+00:00,46.0,Estadio Alberto José Armando,Ciudad de Buenos Aires,451,Boca Juniors,0,0,0,121,Palmeiras,0,0,0
147,13,CONMEBOL Libertadores,2023,Semi-finals,1125289,2023-09-28T00:30:00+00:00,NaN,Estádio do Maracanã,Rio de Janeiro,124,Fluminense,1,1,2,119,Internacional,1,1,2


In [48]:
# lets do one super code
LEAGUE_IDS = [13,11,773]
SEASONS = [2019,2020,2021,2022,2023,2024,2025]
STATUS   = "FT" #,"AET","PEN")

# ——— CONFIG ———
API_KEY    = FB_API_KEY
BASE_URL   = "https://v3.football.api-sports.io/fixtures"
HEADERS = {
    "Accept":           "application/json",
    "x-apisports-key":  API_KEY,
}
# LEAGUE_ID  = 13     # e.g. Premier League
# SEASON     = 2021   # e.g. 2024
# Only completed ("FT","AET","PEN") + in‑progress ("1H","HT","2H","ET","BT","P")
# STATUS   = ("FT","AET","PEN")
CHUNK_SIZE = 20     # max fixture IDs per call :contentReference[oaicite:2]{index=2}

In [49]:
def chunk_list(lst, size):
    """Split list into chunks of at most `size`."""
    return [lst[i:i+size] for i in range(0, len(lst), size)]

def fetch_fixture_ids(league_id, season):
    """Return a list of fixture IDs for the given league/season/status."""
    params = {
        "league": league_id,
        "season": season,
        "status": STATUS
    }
    resp = requests.get(BASE_URL, headers=HEADERS, params=params)
    print(f'running league_id = {league} for season {season}', resp.status_code, resp.headers.get("content-type"))
    #print(resp.text)
    resp.raise_for_status()
    fixtures = resp.json().get("response", [])
    return [f["fixture"]["id"] for f in fixtures if "fixture" in f]

def fetch_fixtures_by_ids(id_group):
    """Fetch full fixture data for a hyphen‑joined group of IDs."""
    ids_param = "-".join(str(i) for i in id_group)
    resp = requests.get(
        BASE_URL,
        headers=HEADERS,
        params={"ids": ids_param}
    )
    resp = requests.get(BASE_URL, headers=HEADERS, params={"ids": ids_param})
    print(resp.status_code, resp.request.url)
    #print(resp.text)   # or resp.json()
    resp.raise_for_status()
    return resp.json().get("response", [])

In [52]:
output_list = []

for league in LEAGUE_IDS:
    loop_list = []
    for season in SEASONS:
        fixture_ids = fetch_fixture_ids(league, season)
        chunks = chunk_list(fixture_ids, CHUNK_SIZE)
        print(f"Will retrieve {len(fixture_ids)} fixtures in {len(chunks)} batches.")

        
        for group in chunks:
            batch = fetch_fixtures_by_ids(group)
            loop_list.extend(batch)
            time.sleep(1)  # crude rate‑limit guard; adjust as needed :contentReference[oaicite:3]{index=3}
    output_list.extend(loop_list)
    
print(f"Retrieved detailed data for {len(output_list)} fixtures.")
        
# Then process with the earlier dict-based extraction
df_rows = []

for match in output_list:
    row = {
        'competition_id': match['league']['id'],
        'competition_name': match['league']['name'],
        'competition_season': match['league']['season'],
        'competition_round': match['league']['round'],
        'fixture_id': match['fixture']['id'],
        'fixture_date': match['fixture']['date'],
        'fixture_venue_id': match['fixture']['venue']['id'],
        'fixture_venue_name': match['fixture']['venue']['name'],
        'fixture_venue_city': match['fixture']['venue']['city'],
        'home_team_id': match['teams']['home']['id'],
        'home_team_name': match['teams']['home']['name'],
        'home_team_h1_goals': match['score']['halftime']['home'],
        'home_team_h2_goals': match['score']['fulltime']['home'] - match['score']['halftime']['home'],
        'home_team_ft_goals': match['score']['fulltime']['home'],
        'away_team_id': match['teams']['away']['id'],
        'away_team_name': match['teams']['away']['name'],
        'away_team_h1_goals': match['score']['halftime']['away'],
        'away_team_h2_goals': match['score']['fulltime']['away'] - match['score']['halftime']['away'],
        'away_team_ft_goals': match['score']['fulltime']['away']
    }
    df_rows.append(row)

complete_df =  pd.DataFrame(df_rows)
        

running league_id = 13 for season 2019 200 application/json
Will retrieve 151 fixtures in 8 batches.
200 https://v3.football.api-sports.io/fixtures?ids=241341-241343-241345-241344-241342-241346-241325-241327-241329-241333-241331-241337-241335-241339-241326-241332-241330-241328-241338-241334
200 https://v3.football.api-sports.io/fixtures?ids=241340-241317-241319-241321-241323-241318-241320-241322-241222-241221-241223-241225-241224-241226-241228-241227-241229-241233-241232-241230
200 https://v3.football.api-sports.io/fixtures?ids=241231-241234-241235-241236-241237-241239-241238-241241-241240-241242-241244-241243-241245-241248-241246-241247-241249-241250-241251-241252
200 https://v3.football.api-sports.io/fixtures?ids=241254-241253-241255-241258-241256-241257-241259-241260-241261-241264-241265-241262-241263-241266-241267-241268-241270-241269-241271-241272
200 https://v3.football.api-sports.io/fixtures?ids=241273-241276-241274-241275-241277-241278-241280-241281-241279-241282-241283-241284-

In [54]:
complete_df.to_csv("south_american_tourney_results.csv")